# 37 — SRE, Incident Response & Ownership

**Role:** Senior AWS DevOps Engineer | **Focus:** Production Operations, Incident Management, Developer Experience, Leadership

---

## Section A: Incident Response

### Q1: Incident Command Structure
**Question:** It's 2 AM and PagerDuty fires. Your main production API is returning 500s for 30% of requests. Walk me through your first 15 minutes.

**Expected Answer:**
- **Minute 0–2**: Acknowledge alert. Check the dashboard. Confirm impact (which services, which customers).
- **Minute 2–5**: Check recent deployments (`git log --since='2 hours ago'`). Correlate with deploy timestamp.
- **Minute 5–10**: If deployment-related → rollback immediately. If not → check infrastructure (RDS CPU, EKS node health, SQS DLQ depth).
- **Minute 10–15**: If not obvious, escalate. Page the service owner. Start a Slack incident channel.
- **Communication**: Post status update every 15 minutes. \"Impact: X% of users affected. Status: Investigating. ETA: Unknown.\"
- **Mitigate first, root-cause later**: Get the service back up. Debug tomorrow.

---

### Q2: Blameless Post-Mortems
**Question:** What makes a good post-mortem? How do you ensure action items actually get implemented?

**Expected Answer:**
- **Template**: Timeline, Impact, Root Cause, Contributing Factors, What Went Well, Action Items.
- **Blameless**: Focus on systems, not people. \"The deployment pipeline lacked a canary stage\" not \"Alice deployed bad code.\"
- **Action items**: Must be SMART (Specific, Measurable, with an Owner and Due Date).
- **Follow-up**: Review action items in weekly engineering meeting. Track completion rate as a team metric.
- **5 Whys**: Drill into root cause. \"Why did the DB crash?\" → \"Connection pool exhausted\" → \"No connection limits configured\" → \"No review checklist for DB config.\"

---

### Q3: Chaos Engineering
**Question:** How do you proactively test system resilience before incidents happen?

**Expected Answer:**
- **AWS Fault Injection Simulator (FIS)**: Inject faults into EC2, EKS, RDS. Simulate AZ failure.
- **Chaos Monkey**: Randomly kill pods/instances to test recovery.
- **Game Days**: Scheduled exercises where the team practices incident response.
- **Litmus Chaos** (K8s-native): Pod kill, network delay, disk fill experiments.
- **Start small**: Begin with non-production. Graduate to production during business hours with rollback ready.
- **Hypothesis-driven**: \"If we kill 1 of 3 EKS nodes, we expect zero user impact due to PDBs and pod anti-affinity.\"

## Section B: Production Operations

### Q4: Disaster Recovery Planning
**Question:** Design a DR strategy for a platform running EKS + RDS in us-east-1. What's your RTO and RPO?

**Expected Answer:**
- **Pilot Light**: Minimal infra in us-west-2 (RDS cross-region read replica, pre-provisioned EKS cluster). RTO: 30–60 min. RPO: ~minutes (async replication lag).
- **Warm Standby**: Scaled-down but running copy. RTO: 10–15 min.
- **Active-Active**: Both regions serve traffic (Route 53 latency-based routing). RTO: ~0. Most expensive.
- **Key steps**: Promote RDS replica → Scale up EKS nodes → Update Route 53 → Verify with health checks.
- **Terraform**: Same IaC deploys to both regions. Region is a variable.
- **Test DR quarterly**: Actually failover. Don't just document it.

---

### Q5: Capacity Planning
**Question:** How do you predict and plan for capacity needs? What metrics do you track?

**Expected Answer:**
- **Trailing metrics**: CPU/memory utilization trends over 30/90 days.
- **Growth projections**: Correlate with business metrics (user signups, API calls/month).
- **Load testing**: k6, Locust, or Artillery to find breaking points. Run monthly.
- **Headroom**: Maintain 30–40% headroom for spikes. Auto-scaling handles the rest.
- **EKS**: Track namespace resource usage vs. quotas. Alert at 70% utilization.
- **RDS**: Monitor connection count trends, storage growth rate, IOPS consumption.
- **Cost forecasting**: AWS Cost Explorer forecast + Infracost for planned Terraform changes.

---

### Q6: Toil Reduction
**Question:** What is \"toil\" in SRE? Give examples and how you've automated them away.

**Expected Answer:**
- **Toil**: Manual, repetitive, automatable, no lasting value work. Examples:
  - Manually rotating secrets → Automated with Secrets Manager rotation Lambda.
  - Manually approving deploys to staging → Auto-approve with passing tests.
  - Manually creating IAM users for new hires → SSO with Okta/Azure AD.
  - Manually cleaning up old Docker images → ECR lifecycle policy.
- **Goal**: Keep toil below 50% of an SRE's time (Google SRE book recommendation).
- **Track**: Log toil tasks in a spreadsheet. Review monthly for automation opportunities.

## Section C: Developer Experience & Ownership

### Q7: Platform as a Product
**Question:** As a platform engineer, your \"customers\" are the developers. How do you measure and improve their experience?

**Expected Answer:**
- **DORA metrics**: Deployment Frequency, Lead Time for Changes, MTTR, Change Failure Rate.
- **Developer survey**: Quarterly NPS on platform tools. What's painful? What's slow?
- **Self-service**: Internal Developer Portal (Backstage) for creating new services, environments.
- **Golden paths**: Opinionated templates (cookiecutter) that include CI/CD, monitoring, IaC out of the box.
- **Documentation**: Runbooks, ADRs (Architecture Decision Records), and examples — not just wiki pages.
- **Feedback loop**: Office hours, Slack channel, retrospective participation.

---

### Q8: Ownership in a Small Team
**Question:** The JD says \"small, high-impact platform team.\" How do you handle on-call, knowledge sharing, and bus factor in a team of 3–5?

**Expected Answer:**
- **On-call**: Rotate weekly. Maximum 1 in 3 weeks. No solo on-call for critical systems.
- **Runbooks**: Every alert has a runbook. If you fix something undocumented, write the runbook before closing the ticket.
- **Pair operations**: Critical infra changes done in pairs (driver + observer).
- **ADRs**: Document WHY decisions were made. New team members can understand context.
- **Bus factor**: No single owner for any system. Cross-train via rotating who works on what.
- **Async-first**: Document in Notion/Confluence, not Slack. Slack messages disappear.

---

### Q9: Technical Decision Making
**Question:** You need to choose between ECS Fargate and EKS for a new microservices platform. How do you make this decision and get buy-in from the team?

**Expected Answer:**
- **RFC/ADR**: Write a 1–2 page document: Context, Options, Pros/Cons, Recommendation.
- **Criteria**: Team expertise, operational overhead, ecosystem needs, portability, cost.
- **Prototype**: Spike both options for 2–3 days. Measure deploy time, debugging experience, cost.
- **Decision**: Present findings to the team. Disagree and commit if needed.
- **Reversibility**: Prefer decisions that are easy to reverse. Containerization (Docker) is the constant; orchestrator can change.
- **Document**: Record the decision, rationale, and expected review date in an ADR.

## Section D: Behavioral Questions

### Q10: Tell Me About a Time...

**Prepare STAR answers for:**
1. **Conflict**: \"...you disagreed with a senior engineer about an architecture decision.\"
2. **Failure**: \"...a deployment you led caused a production outage.\"
3. **Impact**: \"...you significantly improved system reliability or developer productivity.\"
4. **Ownership**: \"...you identified and fixed a problem that wasn't your responsibility.\"
5. **Scaling**: \"...you had to scale a system under unexpected load.\"

**STAR Format:**
- **S**ituation: Set the scene (team size, system, scale).
- **T**ask: What was your specific responsibility?
- **A**ction: What did YOU do? (Use \"I\", not \"we\".)
- **R**esult: Quantify impact (\"Reduced deploy time from 45 min to 8 min\", \"Achieved 99.95% uptime\").